# 利用rating2inter.ipynb中U/I的index对features进行一一对应(meta-text)
- Reindex item feature ID with IDs generated in 0rating2inter.ipynb

In [31]:
import os
import pandas as pd
import random

In [32]:
dataset = 'toys'
os.chdir(f'/home/raozhongtao/MMRec/data/{dataset}')
os.getcwd()

'/home/raozhongtao/MMRec/data/toys'

In [33]:
from collections import defaultdict
# 读取数据
df = pd.read_csv(f'{dataset}.inter', sep='\t')  # 根据实际文件格式调整



# 读取数据
# df = pd.read_csv('your_file.inter', sep='\t')  # 根据实际文件格式调整

# 获取所有唯一的用户ID和项目ID
all_users = df['userID'].unique()
all_items = set(df['itemID'].unique())

# 预先计算每个用户交互过的项目
user_positives = defaultdict(set)
for _, row in df.iterrows():
    user_positives[row['userID']].add(row['itemID'])

# 为每个用户生成负样本
with open('negative_samples.txt', 'w') as f:
    for user in all_users:
        # 获取该用户已经交互过的项目
        positives = user_positives[user]
        
        # 计算负样本集合（差集）
        negatives = list(all_items - positives)
        
        # 确保有足够负样本
        if len(negatives) < 100:
            raise ValueError(f"用户 {user} 的负样本不足100个（只有{len(negatives)}个）")
        
        # 随机选择100个负样本
        sampled_items = random.sample(negatives, 100)
        
        # 写入文件
        f.write(f"{user} " + " ".join(map(str, sampled_items)) + "\n")

print("负样本已保存到 negative_samples.txt")

负样本已保存到 negative_samples.txt


In [34]:
# load item mapping
i_id_mapping = 'i_id_mapping.csv'
df = pd.read_csv(i_id_mapping, sep=',')
print(f'shape: {df.shape}')
df[:4]

shape: (68556, 2)


,item_id,itemID
0,B000E9DPCW,0
1,B000F676D8,1
2,0375829695,2
3,B000AS2AL4,3


In [35]:
import os
import pandas as pd
import gzip, json
meta_file = f'./meta_{dataset}.json.gz'

print('0 Extracting U-I interactions.')

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_file)

print(f'Total records: {meta_df.shape}')
meta_df[:10]

0 Extracting U-I interactions.


Total records: (336072, 9)


,asin,description,title,price,salesRank,imUrl,brand,categories,related
0,0000191639,"Three Dr. Suess' Puzzles: Green Eggs and Ham, ...",Dr. Suess 19163 Dr. Seuss Puzzle 3 Pack Bundle,37.12,{'Toys & Games': 612379},http://ecx.images-amazon.com/images/I/414PLROX...,Dr. Seuss,"[[Toys & Games, Puzzles, Jigsaw Puzzles]]",NaN
1,0005069491,NaN,Nursery Rhymes Felt Book,NaN,{'Toys & Games': 576683},http://ecx.images-amazon.com/images/I/51z4JDBC...,NaN,[[Toys & Games]],NaN
2,0076561046,Learn Fractions Decimals Percents using flash ...,Fraction Decimal Percent Card Deck,NaN,{'Toys & Games': 564211},http://ecx.images-amazon.com/images/I/51ObabPu...,NaN,"[[Toys & Games, Learning & Education, Flash Ca...",{'also_viewed': ['0075728680']}
3,0131358936,"New, Sealed. Fast Shipping with tracking, buy ...",NaN,36.22,{'Software': 8080},http://ecx.images-amazon.com/images/I/51%2B7Ej...,NaN,"[[Toys & Games, Learning & Education, Mathemat...","{'also_bought': ['0321845536', '0078787572'], ..."
4,0133642984,NaN,Algebra 2 California Teacher Center,731.93,{'Toys & Games': 1150291},http://ecx.images-amazon.com/images/I/51VK%2BL...,Prentice Hall,"[[Toys & Games, Learning & Education, Mathemat...",NaN
5,0279515766,NaN,Vintage 1982 Strawberry Shortcake Doll,NaN,{'Toys & Games': 1506850},http://g-ecx.images-amazon.com/images/G/01/x-s...,NaN,"[[Toys & Games, Dolls & Accessories, Dolls]]",NaN
6,0375829695,"A collection of six 48-piece (that is,slightly...",Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,24.82,{'Home &amp; Kitchen': 590975},http://ecx.images-amazon.com/images/I/51Q02ZH6...,Dr. Seuss,"[[Toys & Games, Puzzles, Jigsaw Puzzles]]","{'also_viewed': ['1865036013', 'B004UB2DV4', '..."
7,037585746X,Originally published as two books in slightly ...,Blues Clues On the Go with Blue Color &amp; Ac...,NaN,{'Toys & Games': 417399},http://ecx.images-amazon.com/images/I/517YYdeS...,NaN,"[[Toys & Games, Arts & Crafts, Drawing & Paint...","{'also_bought': ['B0050DRML2', 'B000FCI1Z4', '..."
8,0425066169,Game,Sanctuary: The Thieves' World Boardgame [BOX SET],NaN,{'Toys & Games': 413781},http://ecx.images-amazon.com/images/I/51s4jImJ...,NaN,"[[Toys & Games, Games, Board Games]]",{'also_viewed': ['0912771976']}
9,0439028485,Clifford The Big Red Dog has been a beloved cl...,Clifford The Big Red Dog: 10-Piece Counting &a...,NaN,{'Toys & Games': 427475},http://ecx.images-amazon.com/images/I/41lCoJa0...,NaN,"[[Toys & Games, Learning & Education, Early De...",NaN


In [36]:
# remapping
map_dict = dict(zip(df['item_id'], df['itemID']))

meta_df['itemID'] = meta_df['asin'].map(map_dict)
meta_df.dropna(subset=['itemID'], inplace=True)
meta_df['itemID'] = meta_df['itemID'].astype('int64')
#meta_df['description'] = meta_df['description'].fillna(" ")
meta_df.sort_values(by=['itemID'], inplace=True)

print(f'shape: {meta_df.shape}')
meta_df[:2]

shape: (68556, 10)


,asin,description,title,price,salesRank,imUrl,brand,categories,related,itemID
36710,B000E9DPCW,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Farm Wooden Chunky Puzzle,9.89,{'Toys & Games': 595},http://ecx.images-amazon.com/images/I/51HwF3vH...,Melissa &amp; Doug,"[[Toys & Games, Puzzles]]","{'also_bought': ['B000E9DPVI', 'B000F676D8', '...",0
39441,B000F676D8,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Shapes - Chunky Puzzle,11.19,{'Toys & Games': 1478},http://ecx.images-amazon.com/images/I/511V1HGN...,Melissa &amp; Doug,"[[Toys & Games, Puzzles, Pegged Puzzles]]","{'also_bought': ['B000E9DPCW', 'B000E9DPVI', '...",1


In [37]:
ori_cols = meta_df.columns.tolist()

ret_cols = [ori_cols[-1]] + ori_cols[:-1]
print(f'new column names: {ret_cols}')

new column names: ['itemID', 'asin', 'description', 'title', 'price', 'salesRank', 'imUrl', 'brand', 'categories', 'related']


In [38]:
meta_df[:3]

,asin,description,title,price,salesRank,imUrl,brand,categories,related,itemID
36710,B000E9DPCW,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Farm Wooden Chunky Puzzle,9.89,{'Toys & Games': 595},http://ecx.images-amazon.com/images/I/51HwF3vH...,Melissa &amp; Doug,"[[Toys & Games, Puzzles]]","{'also_bought': ['B000E9DPVI', 'B000F676D8', '...",0
39441,B000F676D8,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Shapes - Chunky Puzzle,11.19,{'Toys & Games': 1478},http://ecx.images-amazon.com/images/I/511V1HGN...,Melissa &amp; Doug,"[[Toys & Games, Puzzles, Pegged Puzzles]]","{'also_bought': ['B000E9DPCW', 'B000E9DPVI', '...",1
6,0375829695,"A collection of six 48-piece (that is,slightly...",Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,24.82,{'Home &amp; Kitchen': 590975},http://ecx.images-amazon.com/images/I/51Q02ZH6...,Dr. Seuss,"[[Toys & Games, Puzzles, Jigsaw Puzzles]]","{'also_viewed': ['1865036013', 'B004UB2DV4', '...",2


In [39]:
ret_df = meta_df[ret_cols]
# dump
ret_df.to_csv(os.path.join('./', f'meta-{dataset}.csv'), index=False)
print('done!')

done!


## Reload

In [40]:
indexed_df = pd.read_csv(f'meta-{dataset}.csv')
print(f'shape: {indexed_df.shape}')
indexed_df[:4]

shape: (68556, 10)


,itemID,asin,description,title,price,salesRank,imUrl,brand,categories,related
0,0,B000E9DPCW,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Farm Wooden Chunky Puzzle,9.89,{'Toys & Games': 595},http://ecx.images-amazon.com/images/I/51HwF3vH...,Melissa &amp; Doug,"[['Toys & Games', 'Puzzles']]","{'also_bought': ['B000E9DPVI', 'B000F676D8', '..."
1,1,B000F676D8,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Shapes - Chunky Puzzle,11.19,{'Toys & Games': 1478},http://ecx.images-amazon.com/images/I/511V1HGN...,Melissa &amp; Doug,"[['Toys & Games', 'Puzzles', 'Pegged Puzzles']]","{'also_bought': ['B000E9DPCW', 'B000E9DPVI', '..."
2,2,0375829695,"A collection of six 48-piece (that is,slightly...",Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,24.82,{'Home &amp; Kitchen': 590975},http://ecx.images-amazon.com/images/I/51Q02ZH6...,Dr. Seuss,"[['Toys & Games', 'Puzzles', 'Jigsaw Puzzles']]","{'also_viewed': ['1865036013', 'B004UB2DV4', '..."
3,3,B000AS2AL4,The sort of learning fun that kids need! Matc...,Melissa &amp; Doug Stack and Sort Board,9.99,{'Toys & Games': 3406},http://ecx.images-amazon.com/images/I/31SS-pAf...,Melissa &amp; Doug,"[['Toys & Games', 'Baby & Toddler Toys', 'Stac...","{'also_bought': ['B000067PWG', 'B00462PTZ4', '..."


In [41]:
## Reload

i_uni = indexed_df['itemID'].unique()

print(f'# of unique items: {len(i_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(i_uni), max(i_uni)))

# of unique items: 68556
min/max of unique learners: 0/68555
